### Belastingen uit RWS waterwebservices

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from toolbox_continu_inzicht import Config

Lees de configuratie in:

Dit leest een configuratie bestand `belasting_rws_config.yaml` in. 
```yaml
GlobalVariables:
    rootdir: "data_sets"
    moments: [ -24, 0, 24, 48 ]

    LoadsWaterwebservicesRWS:
        parameters: [ 'WATHTE' ]

DataAdapter:
    default_options:
        csv:
            sep: ","
    BelastingLocaties:
        type: csv
        path: "belastingen_rws.csv"
    Waterstanden_lokaal:
        type: csv
        path: "hidden_waterstanden.csv"
        sep: ";"
...

```

In [ ]:
data_sets_path = Path.cwd() / "data_sets"
config = Config(config_path=data_sets_path / "belasting_rws_config.yaml")
config.lees_config()

Zet de data adapter klaar en geef deze de configuratie mee 

In [ ]:
from toolbox_continu_inzicht import DataAdapter

data_adapter = DataAdapter(config=config)

Run de module met CSV

In [ ]:
from toolbox_continu_inzicht.loads import LoadsWaterwebservicesRWS

RWS_webservice = LoadsWaterwebservicesRWS(data_adapter=data_adapter)

In [ ]:
RWS_webservice.run(input="BelastingLocaties", output="Waterstanden_lokaal")

In [ ]:
RWS_webservice.df_in

In [ ]:
df_out = RWS_webservice.df_out
df_out

In [ ]:
# split de dataframe in tweeën & process
values_nan = df_out[df_out["value"] == -999].index
for val in values_nan:
    df_out.loc[val, "value"] = np.nan
df_out.sort_index()
df_plot = df_out.set_index(df_out["date_time"])
df_plot = df_plot[
    df_plot["measurement_location_code"]
    == df_out["measurement_location_code"].unique()[0]
]
measurement_index = df_plot["value_type"] == "verwachting"
df_plot_measurements = df_plot[measurement_index][["value"]]
df_plot_forecast = df_plot[~measurement_index][["value"]]

fig, ax = plt.subplots()
df_plot_measurements.plot(color="C0", ax=ax)
df_plot_forecast.plot(color="C1", ax=ax)
ax.legend(["Prediction", "Measurement"]);

```yaml

    Waterstanden_database:
        type: ci_postgresql_measuringstation_to_data
        database: "continuinzicht"
        schema: "continuinzicht_demo_realtime"

```

In [ ]:
from toolbox_continu_inzicht.helpers import calculation_start

start, end = calculation_start(
    data_adapter=data_adapter, output="calculation_start_config", calc_time=None
)
print(f"{start} - {end}")

In [ ]:
# De adapter 'ci_postgresql_calc_status' geeft de huidige rekenstatus en eventuele ui aanpassingen terug
df_in = data_adapter.input("in_ci_status")

# controleer het resultaat op is_calculating = True
is_calculating = df_in[df_in["is_calculating"]]
if len(is_calculating) > 0:
    print("Er wordt al gerekend, toch doorgaan?")
    raise UserWarning("Er wordt al gerekend!")
else:
    print("Er wordt niet gerekend")

In [ ]:
RWS_webservice.run(input="BelastingLocaties", output="Waterstanden_database")

In [ ]:
from toolbox_continu_inzicht.loads import LoadsToMoments

load_moments = LoadsToMoments(data_adapter=data_adapter)
load_moments.run(input="in_measuringstations_table", output="df_moment_waterstanden")

In [ ]:
df_moments = load_moments.df_out.reset_index(drop=False)
df_moments["date_time"] = df_moments["date_time"].astype(object)
# dataframe moments toevoegen aan adapter
data_adapter.set_dataframe_adapter("df_moment_waterstanden", df_moments)
df_moment_waterstanden = data_adapter.input("df_moment_waterstanden")

In [ ]:
condition = data_adapter.input("in_measuringstation_conditions_table")

In [ ]:
data_adapter.input("df_moment_waterstanden")

In [ ]:
from toolbox_continu_inzicht.loads import LoadsClassify

loads_classify = LoadsClassify(data_adapter=data_adapter)
loads_classify.run(
    input=["in_measuringstation_conditions_table", "df_moment_waterstanden"],
    output="out_measuringstation_states_table",
)

In [ ]:
from toolbox_continu_inzicht.helpers import calculation_end

calculation_end(data_adapter=data_adapter, output="calculation_end_config")